In [ ]:
%pip install -q pandas numpy


# Multimodal Integration Audit

Purpose: audit data integration correctness for four selected cBioPortal/TCGA BRCA files.

Files used:

1. `data_clinical_patient.txt`
2. `data_clinical_sample.txt`
3. `data_mrna_seq_v2_rsem_zscores_ref_all_samples.txt`
4. `data_rppa_zscores.txt`

This notebook does **not** train models. It only checks orientation, identifiers, overlaps, missingness, survival columns, modality availability, and the schema of the final patient-level merged dataframe.

In [ ]:
from pathlib import Path
from itertools import combinations
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 120)

DATA_DIR = Path("Dataset") / "brca_tcga_pan_can_atlas_2018"

FILES = {
    "clinical_patient": DATA_DIR / "data_clinical_patient.txt",
    "clinical_sample": DATA_DIR / "data_clinical_sample.txt",
    "rna_zscore": DATA_DIR / "data_mrna_seq_v2_rsem_zscores_ref_all_samples.txt",
    "rppa_zscore": DATA_DIR / "data_rppa_zscores.txt",
}

for name, path in FILES.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing expected file for {name}: {path}")

FILES

## Helper Functions

cBioPortal clinical files often begin with metadata rows prefixed by `#`. The actual tabular header is the first non-comment line. Matrix-style omics files usually have features in rows and samples in columns.

In [ ]:
TCGA_PATIENT_RE = re.compile(r"^TCGA-[A-Z0-9]{2}-[A-Z0-9]{4}$")
TCGA_SAMPLE_RE = re.compile(r"^TCGA-[A-Z0-9]{2}-[A-Z0-9]{4}-\d{2}")

MISSING_TOKENS = {"", "NA", "N/A", "NaN", "nan", "None", "null"}


def is_missing_value(x):
    if pd.isna(x):
        return True
    return str(x).strip() in MISSING_TOKENS


def is_tcga_patient_id(value):
    return isinstance(value, str) and bool(TCGA_PATIENT_RE.match(value.strip()))


def is_tcga_sample_id(value):
    return isinstance(value, str) and bool(TCGA_SAMPLE_RE.match(value.strip()))


def sample_to_patient_id(value):
    """Convert a TCGA sample barcode to a TCGA patient barcode using the first 12 characters."""
    if pd.isna(value):
        return pd.NA
    value = str(value).strip()
    if value.startswith("TCGA-") and len(value) >= 12:
        return value[:12]
    return value


def read_cbio_tsv(path, dtype=None, nrows=None):
    """Read a cBioPortal TSV, skipping metadata/comment rows."""
    return pd.read_csv(
        path,
        sep="\t",
        comment="#",
        dtype=dtype,
        nrows=nrows,
        low_memory=False,
        na_values=["", "NA", "NaN", "nan"],
        keep_default_na=True,
    )


def read_header_only(path):
    return read_cbio_tsv(path, nrows=0).columns.tolist()


def safe_feature_name(value):
    value = "UNKNOWN" if is_missing_value(value) else str(value)
    value = value.strip()
    value = re.sub(r"[^A-Za-z0-9_.|:-]+", "_", value)
    return value.strip("_") or "UNKNOWN"


def make_unique(names):
    seen = {}
    out = []
    for name in names:
        if name not in seen:
            seen[name] = 0
            out.append(name)
        else:
            seen[name] += 1
            out.append(f"{name}__dup{seen[name]}")
    return out


def summarize_set_overlap(named_sets):
    rows = []
    names = list(named_sets)
    for left, right in combinations(names, 2):
        a = named_sets[left]
        b = named_sets[right]
        rows.append({
            "left": left,
            "right": right,
            "left_count": len(a),
            "right_count": len(b),
            "intersection": len(a & b),
            "left_only": len(a - b),
            "right_only": len(b - a),
            "jaccard": len(a & b) / len(a | b) if len(a | b) else np.nan,
        })
    return pd.DataFrame(rows)


## 1. Detect File Orientation

This section infers whether each file is row-oriented, where patients/samples are rows, or matrix-oriented, where features are rows and samples are columns.

In [ ]:
def detect_orientation(path, label):
    header = read_header_only(path)
    preview = read_cbio_tsv(path, nrows=25, dtype=str)

    sample_like_columns = [c for c in header if is_tcga_sample_id(c)]
    patient_like_columns = [c for c in header if is_tcga_patient_id(c)]
    known_row_id_columns = [c for c in ["PATIENT_ID", "SAMPLE_ID", "Tumor_Sample_Barcode", "Sample_Id", "ID"] if c in header]
    feature_id_columns = [c for c in ["Hugo_Symbol", "Entrez_Gene_Id", "Composite.Element.REF"] if c in header]

    row_patient_values = 0
    row_sample_values = 0
    for col in known_row_id_columns:
        values = preview[col].dropna().astype(str)
        row_patient_values += values.map(is_tcga_patient_id).sum()
        row_sample_values += values.map(is_tcga_sample_id).sum()

    if len(sample_like_columns) >= 10:
        orientation = "features_as_rows__samples_as_columns"
        feature_axis = "rows"
        sample_axis = "columns"
        patient_id_source = "derive from sample column names using first 12 barcode characters"
    elif "PATIENT_ID" in header and "SAMPLE_ID" in header:
        orientation = "samples_as_rows__patient_id_column_present"
        feature_axis = "columns"
        sample_axis = "rows"
        patient_id_source = "PATIENT_ID column; validate against SAMPLE_ID first 12 characters"
    elif "PATIENT_ID" in header:
        orientation = "patients_as_rows"
        feature_axis = "columns"
        sample_axis = "not applicable"
        patient_id_source = "PATIENT_ID column"
    else:
        orientation = "unknown_or_metadata"
        feature_axis = "unknown"
        sample_axis = "unknown"
        patient_id_source = "manual inspection needed"

    return {
        "label": label,
        "file": path.name,
        "size_mb": round(path.stat().st_size / (1024 ** 2), 2),
        "n_columns": len(header),
        "preview_rows": len(preview),
        "orientation": orientation,
        "feature_axis": feature_axis,
        "sample_axis": sample_axis,
        "sample_like_columns_in_header": len(sample_like_columns),
        "patient_like_columns_in_header": len(patient_like_columns),
        "known_row_id_columns": ", ".join(known_row_id_columns),
        "feature_id_columns": ", ".join(feature_id_columns),
        "row_patient_values_in_preview": int(row_patient_values),
        "row_sample_values_in_preview": int(row_sample_values),
        "patient_id_source": patient_id_source,
    }


orientation_report = pd.DataFrame([
    detect_orientation(path, label)
    for label, path in FILES.items()
])

orientation_report

## 2. Load Clinical Tables and Extract IDs

In [ ]:
clinical_patient = read_cbio_tsv(FILES["clinical_patient"], dtype=str)
clinical_sample = read_cbio_tsv(FILES["clinical_sample"], dtype=str)

required_patient_cols = {"PATIENT_ID"}
required_sample_cols = {"PATIENT_ID", "SAMPLE_ID"}

missing_patient_cols = required_patient_cols - set(clinical_patient.columns)
missing_sample_cols = required_sample_cols - set(clinical_sample.columns)
if missing_patient_cols:
    raise ValueError(f"Clinical patient file missing columns: {missing_patient_cols}")
if missing_sample_cols:
    raise ValueError(f"Clinical sample file missing columns: {missing_sample_cols}")

clinical_patient["PATIENT_ID"] = clinical_patient["PATIENT_ID"].astype(str).str.strip()
clinical_sample["PATIENT_ID"] = clinical_sample["PATIENT_ID"].astype(str).str.strip()
clinical_sample["SAMPLE_ID"] = clinical_sample["SAMPLE_ID"].astype(str).str.strip()
clinical_sample["PATIENT_ID_FROM_SAMPLE"] = clinical_sample["SAMPLE_ID"].map(sample_to_patient_id)

clinical_id_report = pd.DataFrame([
    {
        "table": "clinical_patient",
        "rows": len(clinical_patient),
        "unique_patient_ids": clinical_patient["PATIENT_ID"].nunique(),
        "duplicate_patient_rows": clinical_patient["PATIENT_ID"].duplicated().sum(),
        "unique_sample_ids": np.nan,
        "sample_patient_mismatch_rows": np.nan,
    },
    {
        "table": "clinical_sample",
        "rows": len(clinical_sample),
        "unique_patient_ids": clinical_sample["PATIENT_ID"].nunique(),
        "duplicate_patient_rows": clinical_sample["PATIENT_ID"].duplicated().sum(),
        "unique_sample_ids": clinical_sample["SAMPLE_ID"].nunique(),
        "sample_patient_mismatch_rows": (clinical_sample["PATIENT_ID"] != clinical_sample["PATIENT_ID_FROM_SAMPLE"]).sum(),
    },
])

clinical_id_report

Rows where `PATIENT_ID` does not match the first 12 characters of `SAMPLE_ID` should be investigated before modeling.

In [ ]:
sample_patient_mismatches = clinical_sample.loc[
    clinical_sample["PATIENT_ID"] != clinical_sample["PATIENT_ID_FROM_SAMPLE"],
    ["PATIENT_ID", "SAMPLE_ID", "PATIENT_ID_FROM_SAMPLE"]
]

sample_patient_mismatches.head(20)

## 3. Extract Omics Sample and Patient IDs

For matrix-style files, the sample IDs are stored in column names. The patient ID is derived from each sample barcode.

In [ ]:
def matrix_sample_id_report(path, label, metadata_columns):
    header = read_header_only(path)
    sample_ids = header[metadata_columns:]
    parsed_patient_ids = [sample_to_patient_id(s) for s in sample_ids]
    sample_suffixes = [str(s)[13:15] if is_tcga_sample_id(str(s)) and len(str(s)) >= 15 else pd.NA for s in sample_ids]

    sample_map = pd.DataFrame({
        "modality": label,
        "sample_id": sample_ids,
        "patient_id": parsed_patient_ids,
        "sample_type_code": sample_suffixes,
    })
    sample_map["is_tcga_sample_id"] = sample_map["sample_id"].map(is_tcga_sample_id)
    sample_map["is_tcga_patient_id"] = sample_map["patient_id"].map(is_tcga_patient_id)

    report = {
        "modality": label,
        "file": path.name,
        "metadata_columns": metadata_columns,
        "sample_columns": len(sample_map),
        "unique_sample_ids": sample_map["sample_id"].nunique(),
        "unique_patient_ids": sample_map["patient_id"].nunique(),
        "duplicate_sample_ids": int(sample_map["sample_id"].duplicated().sum()),
        "duplicate_patient_ids": int(sample_map["patient_id"].duplicated().sum()),
        "invalid_sample_id_columns": int((~sample_map["is_tcga_sample_id"]).sum()),
        "invalid_patient_ids": int((~sample_map["is_tcga_patient_id"]).sum()),
        "sample_type_codes": ", ".join(sorted(sample_map["sample_type_code"].dropna().astype(str).unique())),
    }
    return report, sample_map


rna_id_report, rna_sample_map = matrix_sample_id_report(
    FILES["rna_zscore"],
    label="rna_zscore",
    metadata_columns=2,  # Hugo_Symbol, Entrez_Gene_Id
)

rppa_id_report, rppa_sample_map = matrix_sample_id_report(
    FILES["rppa_zscore"],
    label="rppa_zscore",
    metadata_columns=1,  # Composite.Element.REF
)

omics_id_report = pd.DataFrame([rna_id_report, rppa_id_report])
omics_id_report

In [ ]:
pd.concat([rna_sample_map.head(), rppa_sample_map.head()], ignore_index=True)

## 4. Report Patient Overlap Counts

In [ ]:
patient_sets = {
    "clinical_patient": set(clinical_patient["PATIENT_ID"].dropna().astype(str)),
    "clinical_sample": set(clinical_sample["PATIENT_ID"].dropna().astype(str)),
    "rna_zscore": set(rna_sample_map["patient_id"].dropna().astype(str)),
    "rppa_zscore": set(rppa_sample_map["patient_id"].dropna().astype(str)),
}

patient_set_counts = pd.DataFrame([
    {"modality": name, "unique_patients": len(ids)}
    for name, ids in patient_sets.items()
])

all_four_overlap = set.intersection(*patient_sets.values())
clinical_master_overlap = patient_sets["clinical_patient"] & patient_sets["clinical_sample"] & patient_sets["rna_zscore"] & patient_sets["rppa_zscore"]

display(patient_set_counts)
print(f"Patients present in all four sources: {len(all_four_overlap)}")
print(f"Patients present in clinical patient + sample + RNA + RPPA: {len(clinical_master_overlap)}")

summarize_set_overlap(patient_sets).sort_values(["intersection", "left", "right"], ascending=[False, True, True])

Patients missing from a given source are integration facts, not model-imputation targets yet. This notebook only measures the gaps.

In [ ]:
master_patients = sorted(patient_sets["clinical_patient"])

availability = pd.DataFrame({"PATIENT_ID": master_patients})
for name, ids in patient_sets.items():
    availability[f"has_{name}"] = availability["PATIENT_ID"].isin(ids)

availability["n_modalities_available"] = availability[[c for c in availability.columns if c.startswith("has_")]].sum(axis=1)

availability_summary = pd.DataFrame([
    {
        "field": col,
        "available_patients": int(availability[col].sum()),
        "missing_patients": int((~availability[col]).sum()),
        "missing_percent_of_clinical_master": round((~availability[col]).mean() * 100, 2),
    }
    for col in availability.columns
    if col.startswith("has_")
])

display(availability_summary)
display(availability["n_modalities_available"].value_counts().sort_index().rename_axis("n_available_sources").reset_index(name="patients"))
availability.head()

## 5. Missingness Percentages

Clinical missingness is calculated column-wise. Matrix missingness is calculated cell-wise in chunks to avoid unnecessary memory pressure.

In [ ]:
def column_missingness(df, label):
    missing = df.apply(lambda s: s.map(is_missing_value).mean() * 100)
    return (
        missing.rename("missing_percent")
        .reset_index()
        .rename(columns={"index": "column"})
        .assign(source=label)
        .sort_values("missing_percent", ascending=False)
        [["source", "column", "missing_percent"]]
    )


clinical_missingness = pd.concat([
    column_missingness(clinical_patient, "clinical_patient"),
    column_missingness(clinical_sample, "clinical_sample"),
], ignore_index=True)

clinical_missingness.head(40)

In [ ]:
def matrix_missingness_profile(path, label, metadata_columns, chunksize=1000):
    total_cells = 0
    total_missing = 0
    feature_rows = 0
    sample_missing_counts = None
    sample_total_counts = None
    feature_missing_pcts = []

    for chunk in pd.read_csv(
        path,
        sep="\t",
        comment="#",
        chunksize=chunksize,
        low_memory=False,
        na_values=["", "NA", "NaN", "nan"],
        keep_default_na=True,
    ):
        sample_cols = chunk.columns[metadata_columns:]
        values = chunk.loc[:, sample_cols]
        missing_mask = values.isna()

        if sample_missing_counts is None:
            sample_missing_counts = pd.Series(0, index=sample_cols, dtype="int64")
            sample_total_counts = pd.Series(0, index=sample_cols, dtype="int64")

        sample_missing_counts = sample_missing_counts.add(missing_mask.sum(axis=0), fill_value=0).astype("int64")
        sample_total_counts = sample_total_counts.add(pd.Series(len(chunk), index=sample_cols), fill_value=0).astype("int64")

        row_missing_pct = missing_mask.mean(axis=1) * 100
        feature_missing_pcts.extend(row_missing_pct.tolist())

        total_cells += values.size
        total_missing += int(missing_mask.to_numpy().sum())
        feature_rows += len(chunk)

    sample_missing_pct = (sample_missing_counts / sample_total_counts * 100).astype(float)
    feature_missing_pct = pd.Series(feature_missing_pcts, dtype=float)

    return {
        "modality": label,
        "file": path.name,
        "features": int(feature_rows),
        "samples": int(len(sample_missing_pct)),
        "total_cells": int(total_cells),
        "missing_cells": int(total_missing),
        "missing_cell_percent": round(total_missing / total_cells * 100, 4) if total_cells else np.nan,
        "median_sample_missing_percent": round(sample_missing_pct.median(), 4),
        "max_sample_missing_percent": round(sample_missing_pct.max(), 4),
        "median_feature_missing_percent": round(feature_missing_pct.median(), 4),
        "max_feature_missing_percent": round(feature_missing_pct.max(), 4),
    }


matrix_missingness = pd.DataFrame([
    matrix_missingness_profile(FILES["rna_zscore"], "rna_zscore", metadata_columns=2),
    matrix_missingness_profile(FILES["rppa_zscore"], "rppa_zscore", metadata_columns=1),
])

matrix_missingness

## 6. Identify Survival Columns

Survival status values beginning with `1:` are events. Values beginning with `0:` are censored.

In [ ]:
KNOWN_SURVIVAL_PAIRS = [
    ("OS", "OS_MONTHS", "OS_STATUS"),
    ("DSS", "DSS_MONTHS", "DSS_STATUS"),
    ("DFS", "DFS_MONTHS", "DFS_STATUS"),
    ("PFS", "PFS_MONTHS", "PFS_STATUS"),
]

time_candidates = [c for c in clinical_patient.columns if c.endswith("_MONTHS") or "MONTH" in c.upper()]
status_candidates = [c for c in clinical_patient.columns if c.endswith("_STATUS") or "STATUS" in c.upper()]

survival_rows = []
for endpoint, time_col, status_col in KNOWN_SURVIVAL_PAIRS:
    if time_col not in clinical_patient.columns or status_col not in clinical_patient.columns:
        survival_rows.append({
            "endpoint": endpoint,
            "time_column": time_col,
            "status_column": status_col,
            "present": False,
            "nonmissing_time": np.nan,
            "events": np.nan,
            "censored": np.nan,
            "other_or_missing_status": np.nan,
        })
        continue

    time = clinical_patient[time_col]
    status = clinical_patient[status_col].fillna("").astype(str).str.strip()
    event = status.str.startswith("1:")
    censored = status.str.startswith("0:")
    survival_rows.append({
        "endpoint": endpoint,
        "time_column": time_col,
        "status_column": status_col,
        "present": True,
        "nonmissing_time": int(time.map(lambda x: not is_missing_value(x)).sum()),
        "events": int(event.sum()),
        "censored": int(censored.sum()),
        "other_or_missing_status": int((~event & ~censored).sum()),
    })

survival_report = pd.DataFrame(survival_rows)

print("Time-like candidate columns:", time_candidates)
print("Status-like candidate columns:", status_candidates)
survival_report

## 7. Final Merged Dataframe Schema

The intended merged dataframe is patient-level: one row per `PATIENT_ID`. Omics matrices would be transposed so sample columns become patient rows. This cell builds the schema, not a trained model and not a modeling-ready feature set.

In [ ]:
def get_matrix_features(path, modality, metadata_columns):
    metadata = pd.read_csv(
        path,
        sep="\t",
        comment="#",
        dtype=str,
        usecols=list(range(metadata_columns)),
        low_memory=False,
        na_values=["", "NA", "NaN", "nan"],
        keep_default_na=True,
    ).copy()
    if modality == "rna_zscore":
        hugo = metadata["Hugo_Symbol"].fillna("").astype(str)
        entrez = metadata["Entrez_Gene_Id"].fillna("").astype(str)
        raw_names = [f"{h}|{e}" if h else f"ENTREZ_{e}" for h, e in zip(hugo, entrez)]
        column_names = make_unique([f"RNA__{safe_feature_name(x)}" for x in raw_names])
        original_feature = raw_names
    elif modality == "rppa_zscore":
        raw_names = metadata["Composite.Element.REF"].fillna("").astype(str).tolist()
        column_names = make_unique([f"RPPA__{safe_feature_name(x)}" for x in raw_names])
        original_feature = raw_names
    else:
        raw_names = metadata.iloc[:, 0].fillna("").astype(str).tolist()
        column_names = make_unique([f"{modality.upper()}__{safe_feature_name(x)}" for x in raw_names])
        original_feature = raw_names

    return pd.DataFrame({
        "column_name": column_names,
        "source_file": path.name,
        "modality": modality,
        "role": "feature",
        "merge_key": "PATIENT_ID derived from original sample column barcode",
        "original_feature": original_feature,
    })


schema_parts = []

for col in clinical_patient.columns:
    role = "key" if col == "PATIENT_ID" else "survival" if col in {x for pair in KNOWN_SURVIVAL_PAIRS for x in pair[1:]} else "clinical_patient_feature"
    schema_parts.append({
        "column_name": col,
        "source_file": FILES["clinical_patient"].name,
        "modality": "clinical_patient",
        "role": role,
        "merge_key": "PATIENT_ID",
        "original_feature": col,
    })

for col in clinical_sample.columns:
    if col == "PATIENT_ID":
        continue
    output_col = col if col not in clinical_patient.columns else f"SAMPLE__{col}"
    schema_parts.append({
        "column_name": output_col,
        "source_file": FILES["clinical_sample"].name,
        "modality": "clinical_sample",
        "role": "clinical_sample_feature" if col != "SAMPLE_ID" else "sample_key",
        "merge_key": "PATIENT_ID; validate SAMPLE_ID first 12 characters",
        "original_feature": col,
    })

for availability_col in [c for c in availability.columns if c.startswith("has_")] + ["n_modalities_available"]:
    schema_parts.append({
        "column_name": availability_col,
        "source_file": "computed_in_notebook",
        "modality": "availability_mask",
        "role": "integration_audit_field",
        "merge_key": "PATIENT_ID",
        "original_feature": availability_col,
    })

schema = pd.concat([
    pd.DataFrame(schema_parts),
    get_matrix_features(FILES["rna_zscore"], "rna_zscore", metadata_columns=2),
    get_matrix_features(FILES["rppa_zscore"], "rppa_zscore", metadata_columns=1),
], ignore_index=True)

schema_summary = schema.groupby(["modality", "role"], dropna=False).size().reset_index(name="n_columns")

display(schema_summary)
display(schema.head(25))
display(schema.tail(25))

## 8. Integration Assertions

These assertions should pass before any downstream benchmark construction.

In [ ]:
assert clinical_patient["PATIENT_ID"].is_unique, "Clinical patient table has duplicate PATIENT_ID rows."
assert clinical_sample["SAMPLE_ID"].is_unique, "Clinical sample table has duplicate SAMPLE_ID rows."
assert (clinical_sample["PATIENT_ID"] == clinical_sample["PATIENT_ID_FROM_SAMPLE"]).all(), "Some SAMPLE_ID values do not map back to their PATIENT_ID."
assert len(patient_sets["rna_zscore"]) == rna_sample_map["patient_id"].nunique(), "RNA patient ID extraction inconsistency."
assert len(patient_sets["rppa_zscore"]) == rppa_sample_map["patient_id"].nunique(), "RPPA patient ID extraction inconsistency."
assert survival_report.loc[survival_report["endpoint"].eq("OS"), "present"].iloc[0], "OS survival columns are missing."

print("All integration assertions passed.")
print(f"Final schema columns if fully materialized: {len(schema):,}")
print(f"Patient-level master rows from clinical_patient: {len(master_patients):,}")

## 9. Optional Lightweight Export

This exports only audit artifacts, not a model and not a full high-dimensional merged matrix. Set `WRITE_AUDIT_OUTPUTS = True` if you want CSV copies.

In [ ]:
WRITE_AUDIT_OUTPUTS = False
OUTPUT_DIR = Path("outputs") / "integration_audit"

if WRITE_AUDIT_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    orientation_report.to_csv(OUTPUT_DIR / "orientation_report.csv", index=False)
    clinical_id_report.to_csv(OUTPUT_DIR / "clinical_id_report.csv", index=False)
    omics_id_report.to_csv(OUTPUT_DIR / "omics_id_report.csv", index=False)
    summarize_set_overlap(patient_sets).to_csv(OUTPUT_DIR / "patient_overlap_report.csv", index=False)
    availability.to_csv(OUTPUT_DIR / "modality_availability_matrix.csv", index=False)
    availability_summary.to_csv(OUTPUT_DIR / "modality_availability_summary.csv", index=False)
    clinical_missingness.to_csv(OUTPUT_DIR / "clinical_missingness.csv", index=False)
    matrix_missingness.to_csv(OUTPUT_DIR / "matrix_missingness.csv", index=False)
    survival_report.to_csv(OUTPUT_DIR / "survival_columns_report.csv", index=False)
    schema.to_csv(OUTPUT_DIR / "final_merged_dataframe_schema.csv", index=False)
    print(f"Audit outputs written to: {OUTPUT_DIR.resolve()}")
else:
    print("WRITE_AUDIT_OUTPUTS is False; no files written.")